# Ribosnitch tutorial

This notebook is a **tutorialized** version of the BRIDGE `variant_aware.py` entry point.

- The **code blocks below are copied verbatim** from `variant_aware.py` (i.e., code is consistent with the repository script).
- The notebook adds **step-by-step Markdown explanations**, plus **runnable demos** for parsing/mutation logic.
- End-to-end scoring requires a configured BRIDGE environment (dependencies, transformer path, and `.pth` checkpoints).

## CLI arguments

### Common arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--variation_mode` | `before|after` | yes | Score reference window ('before') or ALT-substituted window ('after'). |
| `--fasta_sequence_path` | `PATH` | yes | Input FASTA containing window sequences. |
| `--variant_out_file` | `PATH` | yes | Output file (appended). |
| `--Transformer_path` | `PATH` | yes | Transformer path used by build_Transformer_embeddings. |
| `--model_save_path` | `PATH` | yes | Directory with BRIDGE checkpoints (*.pth). |
| `--device` | `cuda|cuda:N|cpu` | no | Torch device (default: cuda if available else cpu). |

### Pipeline selection flags
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--gwas` | `-` | no | Force GWAS pipeline (default if no pipeline flag provided). |
| `--ribosnitch/--ribosnitches` | `-` | no | Enable ribosnitch pipeline. |
| `--catalog_variants/--genomic_variants` | `-` | no | Enable catalog-variants pipeline (ClinVar/TCGA/1000G). |

### Ribosnitch-only arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--ribosnitch_after_variation/--ribosnitches_after_variation` | `-` | no | Force ALT substitution in ribosnitch mode regardless of --variation_mode. |
| `--ribosnitch_out_dir/--ribosnitches_out_dir` | `PATH` | no | Root output directory for ribosnitch results (default ./results/ribosnitches). |

## Input FASTA requirements (Ribosnitch)

- Header **must end** with two cell-line tokens (e.g., `HEK293 HepG2`).
- If ALT substitution is used, header must also contain tokens parsable by `parse_variant_block_flexible`.


## Output format (Ribosnitch)

```
<header_without_>\t<checkpoint_stem>\t<float>
```


## Script code

### Imports

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

from __future__ import annotations

import argparse
import logging
import os
import re
import sys
from pathlib import Path
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

repo_root = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(repo_root))

from utils.BRIDGE import BRIDGE
from utils.gen_transformer_embedding import build_Transformer_embeddings
from utils.train_loop import validate_without_sigmoid
from utils.utils import RBPInferDataset
from utils.FeatureEncoding import dealwithdata2
from utils.variant import read_fasta, open_output, parse_variant_block, apply_complement, substitute_base, ModelHub, parse_variant_block_flexible

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Ribosnitch pipeline section

In [4]:
# Ribosnitches pipeline
# ---------------------------------------------------------------------------
def _lazy_import_symbol(module_name: str, symbol_name: str):
    """Import `symbol_name` from `module_name` dynamically (helper for optional deps)."""
    module = importlib.import_module(module_name)
    return getattr(module, symbol_name)


def _extract_cell_lines(header_wo_gt: str) -> Tuple[str, str]:
    """Extract the last two tokens as (cell_line1, cell_line2).

    This matches the user-provided ribosnitches code and assumes the FASTA header
    ends with two cell-line names.
    """
    fields = header_wo_gt.split()
    if len(fields) < 2:
        raise ValueError("Header has fewer than 2 tokens; cannot extract cell lines.")
    return fields[-2], fields[-1]


def _maybe_mutate_sequence_from_header(header: str, seq: str) -> str:
    """Apply ALT substitution based on header tokens (used by ribosnitches-after).

    We use the flexible parser to support headers where the variant token is not
    at a fixed position (e.g. when the header ends with cell-line names).
    """
    var_pos, ref, alt, strand, seq_start = parse_variant_block_flexible(header)

    if strand == "-":
        # The user requested: "only complement, do not reverse" because the variant is centered.
        ref, alt = apply_complement(ref), apply_complement(alt)

    idx0 = var_pos - seq_start
    if idx0 < 0 or idx0 >= len(seq):
        raise ValueError(f"Variant index out of bounds (idx0={idx0}, len={len(seq)})")
    if seq[idx0] != ref:
        raise ValueError(f"Reference base mismatch at idx0={idx0}: seq={seq[idx0]} vs ref={ref}")
    return substitute_base(seq, idx0, alt)


def run_ribosnitches(
    names: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    device: torch.device,
) -> None:
    """Run ribosnitches scoring using the *same* BRIDGE pipeline components as GWAS mode.

    What changes compared to GWAS mode is *only* the checkpoint selection rule:
    we select all `.pth` files under `--model_save_path` that end with either
    `_<cell_line1>.pth` or `_<cell_line2>.pth`, where the two cell lines are taken
    from the last two tokens of each FASTA header.

    Mutation behavior is controlled by:
    - `--variation_mode` (before/after), OR
    - `--ribosnitches_after_variation` (force after)

    Output is appended as TSV:
        <header_without_>  <model_stem>  <score>
    """

    # Decide whether we should substitute ALT in the window
    do_after = bool(args.ribosnitch_after_variation) or (
        bool(args.ribosnitch) and args.variation_mode == "after"
    )

    out_subdir = "after_mut" if do_after else "before_mut"
    out_path = open_output(args.variant_out_file)

    # Loss is only used because validate_without_sigmoid expects it
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2.0, device=device))

    model_dir = Path(args.model_save_path)
    if not model_dir.exists():
        raise FileNotFoundError(f"--model_save_path does not exist: {model_dir}")

    # Reuse BRIDGE checkpoint caching logic from the GWAS branch
    hub = ModelHub(Path(args.Transformer_path), device)

    logging.info("[ribosnitches] Mode=%s | writing to %s", "after" if do_after else "before", out_path)

    with out_path.open("a") as fout:
        for header, seq in zip(names, seqs):
            header_wo_gt = header.lstrip(">")

            # 1) Determine which checkpoints to apply (by cell line suffix)
            try:
                cell_line1, cell_line2 = _extract_cell_lines(header_wo_gt)
            except ValueError as e:
                logging.warning("[ribosnitches] %s -> %s (skip)", header_wo_gt, e)
                continue

            model_files = [
                fn for fn in os.listdir(model_dir)
                if fn.endswith(f"_{cell_line1}.pth") or fn.endswith(f"_{cell_line2}.pth")
            ]
            if not model_files:
                logging.warning("[ribosnitches] No BRIDGE checkpoints for '%s' (skip)", header_wo_gt)
                continue

            # 2) Possibly mutate the input sequence
            try:
                seq_in = _maybe_mutate_sequence_from_header(header, seq) if do_after else seq
            except Exception as e:
                logging.warning("[ribosnitches] %s -> cannot apply variant: %s (skip)", header_wo_gt, e)
                continue

            # 3) Build BRIDGE inputs ONCE per record (then reuse across all checkpoints)
            test_emb, _ = build_Transformer_embeddings(
                sequences=[seq_in],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=1,
                transpose_to_ch_first=True,
            )

            # Keep placeholder tensors consistent with the existing GWAS workflow
            N = int(test_emb.shape[0])
            test_attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            bio_chem = dealwithdata2(seq_in).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=test_emb,
                attn=test_attn,
                struct=struct,
                motif=motif,
                biochem=bio_chem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            # 4) Score with each matching checkpoint
            for filename in model_files:
                stem = Path(filename).stem
                bridge = hub.load_bridge(model_dir, stem)
                if bridge is None:
                    continue

                score = validate_without_sigmoid(bridge, hub.device, loader, criterion).item()
                fout.write(f"{header_wo_gt}\t{stem}\t{score}\n")

    logging.info("[ribosnitches] Done.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def build_argparser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Variant-aware scoring with BRIDGE. Supports three pipelines:\n"
            "  (1) GWAS windows (legacy variant_aware.py behavior; default)\n"
            "  (2) Ribosnitch scoring (BRIDGE; per-record checkpoint selection)\n"
            "  (3) Catalog variants (ClinVar/TCGA/1000G-style FASTA batches; SNVs)\n"
        )
    )

    # ------------------------------------------------------------------
    # Common arguments (shared across pipelines)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--variation_mode",
        choices=["before", "after"],
        required=True,
        help="Score sequences before variation (reference) or after variation (mutated).",
    )
    parser.add_argument("--fasta_sequence_path", required=True, type=Path)
    parser.add_argument("--variant_out_file", required=True, type=Path)
    parser.add_argument("--Transformer_path", required=True, type=Path)
    parser.add_argument("--model_save_path", required=True, type=Path)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------------------------------------------------
    # Pipeline selection flags
    #
    # Backward compatibility:
    # - If you pass none of these flags, we default to GWAS mode.
    # - `--ribosnitches` is accepted as an alias for `--ribosnitch`.
    # - `--genomic_variants` is accepted as an alias for `--catalog_variants`.
    # ------------------------------------------------------------------
    pipe = parser.add_mutually_exclusive_group(required=False)
    pipe.add_argument(
        "--gwas",
        action="store_true",
        help="Force GWAS window scoring (default if no pipeline flag is provided).",
    )
    pipe.add_argument(
        "--ribosnitch",
        "--ribosnitches",
        dest="ribosnitch",
        action="store_true",
        help="Run ribosnitch scoring (BRIDGE).",
    )
    pipe.add_argument(
        "--catalog_variants",
        "--genomic_variants",
        dest="catalog_variants",
        action="store_true",
        help="Run ClinVar/TCGA/1000G-style FASTA batch scoring (SNVs).",
    )

    # ------------------------------------------------------------------
    # Ribosnitch-specific options
    # ------------------------------------------------------------------
    parser.add_argument(
        "--ribosnitch_after_variation",
        "--ribosnitches_after_variation",
        dest="ribosnitch_after_variation",
        action="store_true",
        help="Force ALT substitution for ribosnitch scoring, regardless of --variation_mode.",
    )
    parser.add_argument(
        "--ribosnitch_out_dir",
        "--ribosnitches_out_dir",
        dest="ribosnitch_out_dir",
        type=Path,
        default=Path("./results/ribosnitches"),
        help="Root output directory for ribosnitch results.",
    )

    # ------------------------------------------------------------------
    # Catalog-variants options (also used by genomic_variants.py)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--model_id_strategy",
        choices=["from_header", "from_fasta_stem"],
        default="from_header",
        help=(
            "How to choose checkpoint name for catalog variants. "
            "from_header: <PROTEIN>_<CELL> parsed from header; "
            "from_fasta_stem: use FASTA filename stem."
        ),
    )
    parser.add_argument(
        "--k",
        type=int,
        default=1,
        help="K-mer / stride parameter forwarded to build_Transformer_embeddings (catalog variants branch).",
    )
    parser.add_argument(
        "--pos_weight",
        type=float,
        default=2.0,
        help="Positive class weight for BCEWithLogitsLoss (catalog variants branch).",
    )
    parser.add_argument(
        "--strict_ref_match",
        action="store_true",
        help="If set, skip records when REF/ALT cannot be matched inside the window (catalog variants branch).",
    )
    parser.add_argument(
        "--disable_off_by_one",
        action="store_true",
        help="Disable +/-1 position fallback when locating the SNV inside the window (catalog variants branch).",
    )

    return parser


def main() -> None:
    args = build_argparser().parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    logging.info("Loading FASTA from %s", args.fasta_sequence_path)
    headers, sequences = read_fasta(args.fasta_sequence_path)

    # Pipeline selection validation.
    # - Default: GWAS (for backward compatibility) when no pipeline flag is provided.
    selected = []
    if bool(getattr(args, "gwas", False)):
        selected.append("gwas")
    if bool(getattr(args, "ribosnitch", False)) or bool(getattr(args, "ribosnitch_after_variation", False)):
        selected.append("ribosnitch")
    if bool(getattr(args, "catalog_variants", False)):
        selected.append("catalog_variants")

    if len(selected) > 1:
        raise SystemExit(f"Conflicting pipeline flags: {selected}. Please choose only one.")

    pipeline = selected[0] if selected else "gwas"

    hub = ModelHub(args.Transformer_path, device)

    if pipeline == "ribosnitch":
        logging.info("Running ribosnitch pipeline (%s_variation)", args.variation_mode)
        run_ribosnitches(headers, sequences, args, device)
        
    logging.info("Finished. Results appended to %s", args.variant_out_file)


In [6]:
%%bash
set -euo pipefail

BRIDGE_HOME=../../../
cd "$BRIDGE_HOME"
model_save_path=./results/model

python variant_aware.py \
  --ribosnitch \
  --variation_mode before \
  --fasta_sequence_path ./results/reproducibility/ClinVar_TCGA_1000G_resources/ribosnitches/riboSNitches.fasta \
  --Transformer_path ./RBPformer \
  --model_save_path "$model_save_path" \
  --variant_out_file ./results/reproducibility/ClinVar_TCGA_1000G_resources/ribosnitches/ribosnitch_before.txt \
  --device cuda:3


INFO: Loading FASTA from results/reproducibility/ClinVar_TCGA_1000G_resources/ribosnitches/riboSNitches.fasta
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO: Running ribosnitch pipeline (before_variation)
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.den